In [1]:
import sys
import os

from datetime import datetime
import pandas as pd
import numpy as np

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

In [2]:
from src.ingestion.database import connect_to_database, close_database_connection

# Database connection

In [3]:
cur, conn = connect_to_database()
print(cur, conn)

Successfully connected to the database.
<psycopg.Cursor [no result] [IDLE] (host=127.0.0.1 user=admin database=task_db) at 0x22f65f5cd50> <psycopg.Connection [IDLE] (host=127.0.0.1 user=admin database=task_db) at 0x22f65fc99d0>


# Get data from bookable units table

- This table includes all products and information about the bookings for each day

In [4]:
cur.execute('SELECT * FROM bookable_units')
data = cur.fetchall()

# Columns
column_names = [desc[0] for desc in cur.description]

In [5]:
df_bookable_units = pd.DataFrame(data, columns=column_names)
print(df_bookable_units.shape)

(1327560, 6)


In [6]:
df_bookable_units.head()

,id,product_id,date,count_available_bookings,count_optional_bookings,updated_at
0,42677b926a,XRKO4,2025-11-13,0,0,2026-03-30 11:32:29.751564+00:00
1,7fe820989a,RFJQ1,2025-02-15,0,0,2026-03-30 11:32:29.751564+00:00
2,d02ea1b28d,HVQD1A,2023-10-05,0,0,2026-03-30 11:32:29.751564+00:00
3,ab49f7975b,HVQD3,2027-03-08,0,0,2026-03-30 11:32:29.751564+00:00
4,01929d8f1d,JEQD3I,2025-09-04,0,0,2026-03-30 11:32:29.751564+00:00


# Calculate price recommendation

Formula

- empfohlene Preisanpassung (in %) = 15% * (a - 0.5) * (0.5 + d)
- a = Momentane Auslastung (in %)
- d = (Tage bis zum Reiseantritt / 300)

Example
- Also wenn ein Produkt zum Beispiel 40 Tage vor Reiseantritt zu 75% ausgebucht ist, empfiehlt unsere Funktion eine Preisanpassung von 15% * (0.75 - 0.5) * (0.5 + (40 / 300)) = +2,375%

## Get current date

In [7]:
# Convert date col to datetime for comparison later
df_bookable_units['date'] = pd.to_datetime(df_bookable_units['date'])

In [8]:
current_date = pd.Timestamp.now().normalize()
print(current_date)

2026-03-30 00:00:00


## Get max capacity

- Use maximum of count_available_bookings + count_optional_bookings for now
- A column with total capacity should be requested by customer

In [9]:
df_bookable_units['total_available_bookings'] = df_bookable_units['count_available_bookings'] + df_bookable_units['count_optional_bookings']

df_bookable_units['max_capacity'] = df_bookable_units.groupby('product_id')['total_available_bookings'].transform('max')
df_bookable_units.head()

,id,product_id,date,count_available_bookings,count_optional_bookings,updated_at,total_available_bookings,max_capacity
0,42677b926a,XRKO4,2025-11-13,0,0,2026-03-30 11:32:29.751564+00:00,0,12
1,7fe820989a,RFJQ1,2025-02-15,0,0,2026-03-30 11:32:29.751564+00:00,0,1
2,d02ea1b28d,HVQD1A,2023-10-05,0,0,2026-03-30 11:32:29.751564+00:00,0,0
3,ab49f7975b,HVQD3,2027-03-08,0,0,2026-03-30 11:32:29.751564+00:00,0,0
4,01929d8f1d,JEQD3I,2025-09-04,0,0,2026-03-30 11:32:29.751564+00:00,0,5


# Filter for future dates

In [10]:
df_future = df_bookable_units[df_bookable_units['date'] >= current_date]
df_future.head()

,id,product_id,date,count_available_bookings,count_optional_bookings,updated_at,total_available_bookings,max_capacity
3,ab49f7975b,HVQD3,2027-03-08,0,0,2026-03-30 11:32:29.751564+00:00,0,0
10,a2351bb6fc,RFJA3K,2027-01-28,0,0,2026-03-30 11:32:29.751564+00:00,0,0
11,e1e4383f31,RFZZ4D,2027-11-15,1,0,2026-03-30 11:32:29.751564+00:00,1,1
12,e36e4cfbaa,JEJA4D,2027-04-06,1,0,2026-03-30 11:32:29.751564+00:00,1,1
13,25fcc3be61,HVQD4U,2027-01-06,0,0,2026-03-30 11:32:29.751564+00:00,0,0


In [11]:
print(len(df_future))

489840


## Calculate current occpancy

In [12]:
occupancy_numerator = (df_future['max_capacity'] - df_future['total_available_bookings'])
df_future['occupancy_rate'] = np.where(
    df_future['max_capacity'] > 0, 
    occupancy_numerator / df_future['max_capacity'], 
    0.0
)
df_future.head()

,id,product_id,date,count_available_bookings,count_optional_bookings,updated_at,total_available_bookings,max_capacity,occupancy_rate
3,ab49f7975b,HVQD3,2027-03-08,0,0,2026-03-30 11:32:29.751564+00:00,0,0,0.0
10,a2351bb6fc,RFJA3K,2027-01-28,0,0,2026-03-30 11:32:29.751564+00:00,0,0,0.0
11,e1e4383f31,RFZZ4D,2027-11-15,1,0,2026-03-30 11:32:29.751564+00:00,1,1,0.0
12,e36e4cfbaa,JEJA4D,2027-04-06,1,0,2026-03-30 11:32:29.751564+00:00,1,1,0.0
13,25fcc3be61,HVQD4U,2027-01-06,0,0,2026-03-30 11:32:29.751564+00:00,0,0,0.0


## Calculate days until travel start

In [13]:
df_future['days_until_start'] = (df_future['date'] - current_date).dt.days
df_future['d_factor'] = df_future['days_until_start'] / 300

In [14]:
df_future

,id,product_id,date,count_available_bookings,count_optional_bookings,updated_at,total_available_bookings,max_capacity,occupancy_rate,days_until_start,d_factor
3,ab49f7975b,HVQD3,2027-03-08,0,0,2026-03-30 11:32:29.751564+00:00,0,0,0.000000,343,1.143333
10,a2351bb6fc,RFJA3K,2027-01-28,0,0,2026-03-30 11:32:29.751564+00:00,0,0,0.000000,304,1.013333
11,e1e4383f31,RFZZ4D,2027-11-15,1,0,2026-03-30 11:32:29.751564+00:00,1,1,0.000000,595,1.983333
12,e36e4cfbaa,JEJA4D,2027-04-06,1,0,2026-03-30 11:32:29.751564+00:00,1,1,0.000000,372,1.240000
13,25fcc3be61,HVQD4U,2027-01-06,0,0,2026-03-30 11:32:29.751564+00:00,0,0,0.000000,282,0.940000
...,...,...,...,...,...,...,...,...,...,...,...
1327545,d5cd33021a,IXJA3K,2027-07-17,0,0,2026-03-30 11:32:29.751564+00:00,0,0,0.000000,474,1.580000
1327550,de11342c60,XRJA3,2026-08-26,113,0,2026-03-30 11:32:29.751564+00:00,113,172,0.343023,149,0.496667
1327552,5b94601861,VIYS3Z,2027-05-17,0,0,2026-03-30 11:32:29.751564+00:00,0,0,0.000000,413,1.376667
1327554,13c6202491,RFQD12,2026-09-11,15,0,2026-03-30 11:32:29.751564+00:00,15,15,0.000000,165,0.550000


## Apply formula

- This df should be written to the db

In [15]:
df_future['price_adjustment_pct'] = 0.15 * (df_future['occupancy_rate'] - 0.5) * (0.5 + df_future['d_factor'])

In [16]:
df_future.head()

,id,product_id,date,count_available_bookings,count_optional_bookings,updated_at,total_available_bookings,max_capacity,occupancy_rate,days_until_start,d_factor,price_adjustment_pct
3,ab49f7975b,HVQD3,2027-03-08,0,0,2026-03-30 11:32:29.751564+00:00,0,0,0.0,343,1.143333,-0.12325
10,a2351bb6fc,RFJA3K,2027-01-28,0,0,2026-03-30 11:32:29.751564+00:00,0,0,0.0,304,1.013333,-0.11350
11,e1e4383f31,RFZZ4D,2027-11-15,1,0,2026-03-30 11:32:29.751564+00:00,1,1,0.0,595,1.983333,-0.18625
12,e36e4cfbaa,JEJA4D,2027-04-06,1,0,2026-03-30 11:32:29.751564+00:00,1,1,0.0,372,1.240000,-0.13050
13,25fcc3be61,HVQD4U,2027-01-06,0,0,2026-03-30 11:32:29.751564+00:00,0,0,0.0,282,0.940000,-0.10800


## Get max price adjustment

- This should be the output of the third endpoint

In [17]:
max_price_adjustment = df_future['price_adjustment_pct'].max()

max_recommendation = df_future[df_future['price_adjustment_pct'] == max_price_adjustment]
max_recommendation.iloc[0]

id                                                b3445ac35b
product_id                                            IXSE3W
date                                     2027-12-17 00:00:00
count_available_bookings                                   0
count_optional_bookings                                    0
updated_at                  2026-03-30 11:32:29.751564+00:00
total_available_bookings                                   0
max_capacity                                               7
occupancy_rate                                           1.0
days_until_start                                         627
d_factor                                                2.09
price_adjustment_pct                                 0.19425
Name: 42030, dtype: object